In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys, pathlib

# adjust this to your repo root if different
repo_root = pathlib.Path.cwd()

repo_root = repo_root.parent.parent

sys.path.insert(0, str(repo_root))
print('Inserted repo root into sys.path:', repo_root)

Inserted repo root into sys.path: /home/ubuntu/qecsim


In [4]:
from src.codes.surface_code_rotated.builder import SurfaceBuilder
from src.core.data_models import NoiseParameters
from src.codes.lattice_surgery.builder import SurgeryBuilder
import itertools
from tqdm import tqdm
import sinter
import math
import os
from matplotlib import pyplot as plt


In [ ]:
# Intial Setup Values
geom = [10**(math.log10(1e-5) + i*(math.log10(1e-3) - math.log10(1e-5))/9) for i in range(10)]
lin = [1.1e-3 + i*(0.015 - 1.1e-3)/39 for i in range(40)]
p_values = geom + lin
distances = [3, 5, 7]
combinations = list(itertools.product(distances, p_values))

# Combinations to simulate
cases = [
    ("X+", "X"), 
    ("Y+", "Y"), 
    ("Z0", "Z")
]

for curr_config in cases:
    # Build the tasks
    tasks = [
        sinter.Task(
            circuit=SurfaceBuilder(
                distance=d,
                state_init= curr_config[0],
                log_obs= curr_config[1],
                noise=NoiseParameters(
                    before_round_depol=noise,
                    before_m_flip_prob=noise,
                    after_r_flip=noise,
                    after_c_depol_prob=noise
                )
            ).build_circuit(),
            json_metadata={'d': d, 'p': noise, 'config': curr_config},
        )
        for d, noise in tqdm(combinations, desc=f"Building {curr_config}")
    ]

    # Set Filename
    filename = f"../data/stats_memory_{'_'.join(curr_config)}.csv"

    # Hand them off to sinter
    collected_stats = sinter.collect(
        num_workers=os.cpu_count(),
        tasks=tasks,
        decoders=['pymatching'],
        max_shots=1_000_000,
        max_errors=10_000,
        print_progress=True,
        save_resume_filepath= filename
    )

In [ ]:
cases = [
    ("X+", "X"), 
    ("Y+", "Y"), 
    ("Z0", "Z")
]

for curr_config in cases:
    config_suffix = '_'.join(curr_config)
    csv_filename = f"../data/stats_memory_{config_suffix}.csv"
    plot_filename = f"../data/plot_memory_{config_suffix}.pdf"

    # Check if the data file exists before trying to plot
    if not os.path.exists(csv_filename):
        print(f"Skipping {csv_filename}: File not found.")
        continue

    # 1. Load the stats from the CSV
    stats = sinter.read_stats_from_csv_files(csv_filename)

    # 2. Create the figure
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    # 3. Use sinter's plotting utility
    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
        # Optional: ensure consistent markers for distances
        # marker_func=lambda s: {3: 'o', 5: 's', 7: 'D'}.get(s.json_metadata['d'], 'v')
    )

    # 4. Professional Styling
    ax.loglog()
    ax.set_title(f"Surface Memory (Circuit)-Noise: {curr_config}")
    ax.set_xlabel("Physical Error Rate $p$")
    ax.set_ylabel("Logical Error Rate $p_L$")
    ax.grid(which='major', linestyle='-', alpha=0.6)
    ax.grid(which='minor', linestyle=':', alpha=0.3)
    ax.legend(title="Distance $d$", loc='lower right')

    # 5. Save the plot
    fig.set_dpi(300)
    plt.tight_layout()
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.close(fig)  # Close to free up memory during the loop

    print(f"Successfully saved: {plot_filename}")

In [ ]:
# Intial Setup Values
geom = [10**(math.log10(1e-5) + i*(math.log10(1e-3) - math.log10(1e-5))/9) for i in range(10)]
lin = [1.1e-3 + i*(0.015 - 1.1e-3)/39 for i in range(40)]
p_values = geom + lin
distances = [3, 5, 7]
combinations = list(itertools.product(distances, p_values))

# Combinations to simulate
cases = [
    ("I0", "Z0", "Z", "Z"), 
    ("Z0", "I0", "Z", "I"), 
    ("X+", "I0", "X", "X"), 
    ("I0", "X+", "I", "X")
]

for curr_config in cases:
    # Build the tasks
    tasks = [
        sinter.Task(
            circuit=SurgeryBuilder(
                distance=d,
                control_state_init=curr_config[0],
                target_state_init=curr_config[1],
                control_measure_basis=curr_config[2],
                target_measure_basis=curr_config[3],
                noise=NoiseParameters(
                    before_round_depol=noise,
                    before_m_flip_prob=noise,
                    after_r_flip=noise,
                    after_c_depol_prob=noise
                )
            ).build_circuit(),
            json_metadata={'d': d, 'p': noise, 'config': curr_config},
        )
        for d, noise in tqdm(combinations, desc=f"Building {curr_config}")
    ]

    # Set Filename
    filename = f"../data/stats_{'_'.join(curr_config)}.csv"

    # Hand them off to sinter
    collected_stats = sinter.collect(
        num_workers=os.cpu_count(),
        tasks=tasks,
        decoders=['pymatching'],
        max_shots=1_000_000,
        max_errors=10_000,
        print_progress=True,
        save_resume_filepath= filename
    )

In [ ]:
cases = [
    ("I0", "Z0", "Z", "Z"), 
    ("Z0", "I0", "Z", "I"), 
    ("X+", "I0", "X", "X"), 
    ("I0", "X+", "I", "X")
]

for curr_config in cases:
    config_suffix = '_'.join(curr_config)
    csv_filename = f"../data/stats_{config_suffix}.csv"
    plot_filename = f"../data/plot_{config_suffix}.pdf"

    # Check if the data file exists before trying to plot
    if not os.path.exists(csv_filename):
        print(f"Skipping {csv_filename}: File not found.")
        continue

    # 1. Load the stats from the CSV
    stats = sinter.read_stats_from_csv_files(csv_filename)

    # 2. Create the figure
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    # 3. Use sinter's plotting utility
    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
        # Optional: ensure consistent markers for distances
        # marker_func=lambda s: {3: 'o', 5: 's', 7: 'D'}.get(s.json_metadata['d'], 'v')
    )

    # 4. Professional Styling
    ax.loglog()
    ax.set_title(f"Lattice Surgery (Circuit)-Noise: {curr_config}")
    ax.set_xlabel("Physical Error Rate $p$")
    ax.set_ylabel("Logical Error Rate $p_L$")
    ax.grid(which='major', linestyle='-', alpha=0.6)
    ax.grid(which='minor', linestyle=':', alpha=0.3)
    ax.legend(title="Distance $d$", loc='lower right')

    # 5. Save the plot
    fig.set_dpi(300)
    plt.tight_layout()
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.close(fig)  # Close to free up memory during the loop

    print(f"Successfully saved: {plot_filename}")